# Formula 1 Race State Visualizer & Dataset Generator

This notebook provides a **live, visual representation** of race state evolution across laps:
- **Section 1: Grid Summary (2 Tables)**: Real-time animated updates for **Table 1 (Classification & Timing)** and **Table 2 (Tyre, Strategy & Telemetry)**.
- **Section 2: Driver Detail Card**: Real-time animated updates for an individual driver telemetry & strategy card (e.g., `VER`, `HAM`, `NOR`).
- **Persistent JSON Storage**: Lap-by-lap snapshots are exported to a dedicated JSON file (overwritten per run to prevent duplicate clutter) for downstream machine learning and strategy models.

In [ ]:
# Auto-reload local modules when updated on disk
%load_ext autoreload
%autoreload 2

# Setup paths and environment
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CSV_PATH = Path('data_fastf1_v1/laps/2024/British_Grand_Prix.csv')
OUTPUT_JSON = Path('data_fastf1_v1/race_state_snapshots_british_gp_2024.json')

print('Target CSV:', CSV_PATH.resolve())
print('Target JSON Export:', OUTPUT_JSON.resolve())

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Target CSV: D:\F1-SIS-Capstone-Project\data_fastf1_v1\laps\2024\British_Grand_Prix.csv
Target JSON Export: D:\F1-SIS-Capstone-Project\data_fastf1_v1\race_state_snapshots_british_gp_2024.json


## Data Cleaning

The cells below apply **four cleaning steps** to the raw FastF1 CSV before any lap is committed to the race state.
The cleaned rows are stored in `cleaned_rows` and `grouped_clean`, which are used by all downstream replay sections instead of the raw CSV rows.

| Step | What it does |
|---|---|
| **1. Quality-flag filter** | Drops rows where `IsAccurate != True` or `Deleted == True` or `FastF1Generated == True` |
| **2. Boundary check** | Drops rows where `LapTimeSeconds` is outside the physical range `[50 s, 200 s]` |
| **3. Pace-contamination flag** | Tags pit in-laps, pit out-laps, and SC/VSC laps as `_exclude_from_pace = True` so rolling averages skip them |
| **4. Retirement detection** | After all rows are grouped by lap, marks any driver as retired if they stop appearing in the timing data while the race continues |

In [ ]:
# ==========================================================================
# Data Cleaning — Step 1: Quality-Flag Filter
#                 Step 2: Lap-Time Boundary Check
# ==========================================================================
import csv
from src.race_state.models import safe_float, safe_bool, parse_lap_number

# Physical lap-time bounds (seconds).
# Laps outside this window are corrupted or non-representative.
LAP_TIME_MIN_S = 50.0
LAP_TIME_MAX_S = 200.0

raw_rows = []
with open(CSV_PATH, newline='', encoding='utf-8') as fh:
    raw_rows = list(csv.DictReader(fh))

dropped_quality  = 0
dropped_boundary = 0
cleaned_rows     = []

for row in raw_rows:
    # --- Step 1: Quality flags -------------------------------------------
    is_accurate      = safe_bool(row.get('IsAccurate'))       # must be True
    is_deleted       = safe_bool(row.get('Deleted'))           # must be False / None
    is_ff1_generated = safe_bool(row.get('FastF1Generated'))   # must be False / None

    if is_accurate is False:         # explicit False -> corrupted timing
        dropped_quality += 1
        continue
    if is_deleted is True:           # lap deleted (e.g. track-limit violation)
        dropped_quality += 1
        continue
    if is_ff1_generated is True:     # synthetic / interpolated record
        dropped_quality += 1
        continue

    # --- Step 2: Lap-time boundary check ---------------------------------
    lap_time_s = safe_float(row.get('LapTimeSeconds'))
    if lap_time_s is not None:
        if lap_time_s < LAP_TIME_MIN_S or lap_time_s > LAP_TIME_MAX_S:
            dropped_boundary += 1
            continue

    cleaned_rows.append(row)

total_raw     = len(raw_rows)
total_cleaned = len(cleaned_rows)
print(f'Raw rows            : {total_raw}')
print(f'Dropped (quality)   : {dropped_quality}')
print(f'Dropped (boundary)  : {dropped_boundary}')
print(f'Rows after cleaning : {total_cleaned}')
print(f'Retention rate      : {100 * total_cleaned / total_raw:.1f}%')

In [ ]:
# ==========================================================================
# Data Cleaning — Step 3: Pace-Contamination Flagging
#                 Step 4: Retirement Detection
# ==========================================================================

# Group cleaned rows by lap number.
# `grouped_clean` replaces the raw grouping used in all sections below.
grouped_clean: dict = {}
for row in cleaned_rows:
    lap = parse_lap_number(row.get('LapNumber'))
    if lap is not None:
        grouped_clean.setdefault(lap, []).append(row)

sorted_laps_clean = sorted(grouped_clean.keys())

# --- Step 3: Tag rows that must NOT contribute to rolling pace averages --
# Pit in-laps, pit out-laps, and Safety Car / VSC laps are excluded because
# their lap times do not reflect true representative race pace.
pace_excluded = 0
for lap, rows in grouped_clean.items():
    for row in rows:
        is_pit_in  = safe_float(row.get('PitInTimeSeconds'))  is not None
        is_pit_out = safe_float(row.get('PitOutTimeSeconds')) is not None
        has_sc     = safe_bool(row.get('HasSafetyCar')) is True
        has_vsc    = safe_bool(row.get('HasVSC'))        is True

        # Internal marker read by downstream consumers to skip this row
        # when computing rolling_3_lap_avg and rolling_5_lap_avg.
        row['_exclude_from_pace'] = (
            is_pit_in or is_pit_out or has_sc or has_vsc
        )
        if row['_exclude_from_pace']:
            pace_excluded += 1

# --- Step 4: Retirement detection ----------------------------------------
# If a driver disappears from timing for >= RETIREMENT_GAP consecutive laps
# while the race is still running, they are considered retired.
RETIREMENT_GAP = 2   # consecutive laps of silence = retired

drivers_per_lap: dict = {
    lap: {
        row.get('Driver', '').strip().upper()
        for row in rows
        if row.get('Driver')
    }
    for lap, rows in grouped_clean.items()
}

all_drivers = set().union(*drivers_per_lap.values())
retired_on: dict = {}   # driver -> lap where retirement was detected

for driver in all_drivers:
    consecutive_absent = 0
    for lap in sorted_laps_clean:
        if driver in drivers_per_lap[lap]:
            consecutive_absent = 0
        else:
            consecutive_absent += 1
            if consecutive_absent >= RETIREMENT_GAP and driver not in retired_on:
                retired_on[driver] = lap

print(f'Laps after cleaning        : {len(sorted_laps_clean)}')
print(f'Rows flagged (pace-excl.)  : {pace_excluded}')
print(f'Drivers detected as retired: {len(retired_on)}')
if retired_on:
    for drv, lap in sorted(retired_on.items(), key=lambda x: x[1]):
        print(f'   {drv} — retirement detected on lap {lap}')

##  Section 1: Live Grid Summary (Timing + Strategy Tables)

Run the cell below to watch the race unfold lap by lap. Both tables dynamically update right here in the notebook at human-readable speed (`LAP_DELAY_SECONDS`), showing position changes, delta gaps, tyre life, and rolling pace as laps advance.

All lap-by-lap states are exported to the JSON file at the end of the run.

In [ ]:
import time
import json
from IPython.display import clear_output, display, HTML

from src.race_state.manager import RaceStateManager
from src.race_state.replay import load_csv_rows
from src.race_state.models import parse_lap_number
from src.race_state.notebook_display import render_grid_summary_html

# ==========================================================================
# Animation & Replay Configuration
# ==========================================================================
LAP_DELAY_SECONDS = 0.65  # Seconds per lap to clearly observe changes (e.g., 0.5 to 1.0)
MAX_LAPS = None           # Set to an int (e.g. 15) to preview fewer laps, or None for all laps
TOP_N_DRIVERS = 10        # Top N classified drivers to show in tables (e.g. 10 or 20)

# 1. Load CSV data and group by lap
rows = list(load_csv_rows(CSV_PATH))
grouped = {}
for r in rows:
    lap = parse_lap_number(r.get('LapNumber'))
    if lap is not None:
        grouped.setdefault(lap, []).append(r)

sorted_laps = sorted(grouped.keys())
if MAX_LAPS:
    sorted_laps = sorted_laps[:MAX_LAPS]

# 2. Initialize engine and storage list
manager = RaceStateManager()
race_snapshots = []

# 3. Step through laps: clear and re-display for smooth, visible live updates
for lap in sorted_laps:
    manager.update_from_rows(grouped[lap])
    state = manager.commit_lap(lap)
    race_snapshots.append(state.to_dict())

    # Generate rich HTML tables
    html_content = render_grid_summary_html(state, limit=TOP_N_DRIVERS)

    # clear_output(wait=True) ensures seamless transition without screen flicker
    clear_output(wait=True)
    display(HTML(html_content))
    time.sleep(LAP_DELAY_SECONDS)

# 4. Save JSON file (overwriting any previous run to keep clean data)
OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(race_snapshots, f, indent=2)

print(f' Replay complete! Saved {len(race_snapshots)} lap snapshots to {OUTPUT_JSON.name}')

Pos,Chg,Driver,Team,Last Lap,Total Time,S1,S2,S3,Gap (s),Int (s),Behind (s)
1,-,HAM,Mercedes,1:30.146,2:20:42.803,29.30,36.13,24.71,,,1.534
2,-,VER,Red Bull Racing,1:29.089,2:20:44.268,29.15,35.71,24.23,1.534,1.534,6.035
3,-,NOR,McLaren,1:30.913,2:20:50.350,29.63,36.65,24.64,7.569,6.035,4.918
4,-,PIA,McLaren,1:29.928,2:20:55.232,29.09,36.00,24.84,12.487,4.918,34.924
5,-,SAI,Ferrari,1:28.293,2:21:30.121,28.55,35.62,24.12,47.411,34.924,8.404
6,-,HUL,Haas F1 Team,1:30.847,2:21:38.525,29.43,36.37,25.04,55.722,8.404,0.847
7,-,STR,Aston Martin,1:30.053,2:21:39.372,29.02,36.30,24.73,56.569,0.847,7.030
8,-,ALO,Aston Martin,1:31.430,2:21:46.380,29.31,37.16,24.96,63.642,7.030,4.810
9,-,ALB,Williams,1:29.718,2:21:51.190,29.12,36.19,24.41,68.387,4.810,10.916
10,-,TSU,RB,1:32.090,2:22:02.106,29.55,36.95,25.59,79.303,10.916,9.657


 Replay complete! Saved 52 lap snapshots to race_state_snapshots_british_gp_2024.json


##  Section 2: Driver Detail Card

Run the cell below to focus on an individual driver (e.g. `VER`, `HAM`, `NOR`, `PIA`, `SAI`). The telemetry card updates lap by lap showing tyre wear, delta gaps, sectors, speeds, and rolling pace.

In [ ]:
import time
from IPython.display import clear_output, display, HTML

from src.race_state.manager import RaceStateManager
from src.race_state.replay import load_csv_rows
from src.race_state.models import parse_lap_number
from src.race_state.notebook_display import render_driver_card_html

# ==========================================================================
# Driver Replay Configuration
# ==========================================================================
SELECTED_DRIVER = 'VER'   # Change to any 3-letter driver code: HAM, NOR, PIA, SAI, etc.
LAP_DELAY_SECONDS = 0.5   # Delay between laps in seconds
MAX_LAPS = None           # Set integer or None for all laps

rows = list(load_csv_rows(CSV_PATH))
grouped = {}
for r in rows:
    lap = parse_lap_number(r.get('LapNumber'))
    if lap is not None:
        grouped.setdefault(lap, []).append(r)

sorted_laps = sorted(grouped.keys())
if MAX_LAPS:
    sorted_laps = sorted_laps[:MAX_LAPS]

manager = RaceStateManager()

for lap in sorted_laps:
    manager.update_from_rows(grouped[lap])
    state = manager.commit_lap(lap)

    card_html = render_driver_card_html(state, SELECTED_DRIVER)
    clear_output(wait=True)
    display(HTML(card_html))
    time.sleep(LAP_DELAY_SECONDS)

print(f' Telemetry replay complete for {SELECTED_DRIVER} across {len(sorted_laps)} laps!')

 Telemetry replay complete for VER across 52 laps!
